In [2]:
%pip install scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.22.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.2 MB 6.0 MB/s eta 0:00:02
   ----------- ---------------------------- 2.4/8.2 MB 6.9 MB/s eta 0:00:01
   -------------------- ------------------- 4.2/8.2 MB 7.5 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.2 MB 8.2 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 8.3 MB/s  0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached narwhals-2.22.1-py3-none-any.whl (454 kB)
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   -- ------------------------------------- 2.1/36.6 MB 11.2 MB/s eta 0:00:04
   ---- ----------------------------------- 4.5/36.6 MB 11.6 MB/s eta 0:00:03
   ------- -------------

In [3]:
# Cell 1 — notebooks/02_preprocessing.ipynb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

df = pd.read_csv('../data/raw/StudentPerformanceFactors.csv')
df.shape

(6607, 20)

3.2 — Handle Missing Values

In [4]:
# Check missing values again (ground truth before we act)
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing)

Teacher_Quality             78
Parental_Education_Level    90
Distance_from_Home          67
dtype: int64


Decision logic

In [5]:
# For numeric columns with missing values: fill with median
# Why median, not mean: median is robust to outliers (Phase 2 boxplots
# may have shown some skew), mean gets pulled by extreme values
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled {col} missing values with median: {median_val}")

# For categorical columns with missing values: fill with mode (most frequent category)
# Why mode: it's the "most likely" category for a missing entry, a reasonable default
categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"Filled {col} missing values with mode: {mode_val}")

# Verify no missing values remain
print("\nRemaining missing values:", df.isnull().sum().sum())

Filled Teacher_Quality missing values with mode: Medium
Filled Parental_Education_Level missing values with mode: High School
Filled Distance_from_Home missing values with mode: Near

Remaining missing values: 0


C:\Users\MC USER\AppData\Local\Temp\ipykernel_27624\3431701833.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include='object').columns


3.3 — Remove Duplicates

In [6]:
print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after:", df.duplicated().sum())
print("New shape:", df.shape)

Duplicates before: 0
Duplicates after: 0
New shape: (6607, 20)


3.4 — Handle Outliers

In [7]:
# We use the IQR (Interquartile Range) method — standard, explainable, and
# doesn't require assuming a normal distribution

def get_outlier_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

outlier_report = {}
for col in numeric_cols:
    if col == 'Exam_Score':
        continue  # we generally don't touch the target itself
    lower, upper = get_outlier_bounds(df[col])
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outlier_report[col] = len(outliers)

print(pd.Series(outlier_report).sort_values(ascending=False))

Tutoring_Sessions    430
Hours_Studied         43
Attendance             0
Sleep_Hours            0
Previous_Scores        0
Physical_Activity      0
dtype: int64


In [8]:
# Decision: CAP outliers instead of deleting them (capping = "winsorizing")
# Why cap instead of drop: we keep the row (and its other valid feature values)
# while preventing the extreme single value from distorting the model
for col in numeric_cols:
    if col == 'Exam_Score':
        continue
    lower, upper = get_outlier_bounds(df[col])
    df[col] = np.clip(df[col], lower, upper)

print("Outlier capping complete.")

Outlier capping complete.


3.5 — Encode Categorical Variables

In [9]:
df.select_dtypes(include='object').nunique()

C:\Users\MC USER\AppData\Local\Temp\ipykernel_27624\3404847108.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include='object').nunique()


Parental_Involvement          3
Access_to_Resources           3
Extracurricular_Activities    2
Motivation_Level              3
Internet_Access               2
Family_Income                 3
Teacher_Quality               3
School_Type                   2
Peer_Influence                3
Learning_Disabilities         2
Parental_Education_Level      3
Distance_from_Home            3
Gender                        2
dtype: int64

In [31]:
# Ordinal columns — these have a natural Low < Medium < High order
ordinal_cols = {
    'Parental_Involvement': ['Low', 'Medium', 'High'],
    'Access_to_Resources': ['Low', 'Medium', 'High'],
    'Motivation_Level': ['Low', 'Medium', 'High'],
    'Family_Income': ['Low', 'Medium', 'High'],
    'Teacher_Quality': ['Low', 'Medium', 'High'],
    'Parental_Education_Level': ['High School', 'College', 'Postgraduate'],
    'Distance_from_Home': ['Near', 'Moderate', 'Far']
}

for col, order in ordinal_cols.items():
    if col in df.columns:
        mapping = {category: i for i, category in enumerate(order)}
        df[col] = df[col].map(mapping)
        print(f"Encoded {col}: {mapping}")

Encoded Parental_Involvement: {'Low': 0, 'Medium': 1, 'High': 2}
Encoded Access_to_Resources: {'Low': 0, 'Medium': 1, 'High': 2}
Encoded Motivation_Level: {'Low': 0, 'Medium': 1, 'High': 2}
Encoded Family_Income: {'Low': 0, 'Medium': 1, 'High': 2}
Encoded Teacher_Quality: {'Low': 0, 'Medium': 1, 'High': 2}
Encoded Parental_Education_Level: {'High School': 0, 'College': 1, 'Postgraduate': 2}
Encoded Distance_from_Home: {'Near': 0, 'Moderate': 1, 'Far': 2}


In [24]:
# Nominal columns — no inherent order, use one-hot encoding
nominal_cols = ['Extracurricular_Activities', 'Internet_Access', 
                 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Gender']

nominal_cols = [c for c in nominal_cols if c in df.columns]

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)
print("Shape after one-hot encoding:", df.shape)
df.head()

Shape after one-hot encoding: (6607, 21)


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Sleep_Hours,Previous_Scores,Motivation_Level,Tutoring_Sessions,Family_Income,Teacher_Quality,...,Parental_Education_Level,Distance_from_Home,Exam_Score,Extracurricular_Activities_Yes,Internet_Access_Yes,School_Type_Public,Peer_Influence_Neutral,Peer_Influence_Positive,Learning_Disabilities_Yes,Gender_Male
0,23,84,NaN,NaN,7,73,NaN,0.0,NaN,NaN,...,NaN,NaN,67,False,True,True,False,True,False,True
1,19,64,NaN,NaN,8,59,NaN,2.0,NaN,NaN,...,NaN,NaN,61,False,True,True,False,False,False,False
2,24,98,NaN,NaN,7,91,NaN,2.0,NaN,NaN,...,NaN,NaN,74,True,True,True,True,False,False,True
3,29,89,NaN,NaN,8,98,NaN,1.0,NaN,NaN,...,NaN,NaN,71,True,True,True,False,False,False,True
4,19,92,NaN,NaN,6,65,NaN,3.0,NaN,NaN,...,NaN,NaN,70,True,True,True,True,False,False,False


3.6 — Feature Selection

In [25]:
# Re-check correlation with target now that everything is numeric
correlations = df.corr()['Exam_Score'].sort_values(ascending=False)
print(correlations)

Exam_Score                        1.000000
Attendance                        0.581072
Hours_Studied                     0.445387
Previous_Scores                   0.175079
Tutoring_Sessions                 0.150742
Peer_Influence_Positive           0.081217
Extracurricular_Activities_Yes    0.064382
Internet_Access_Yes               0.051475
Physical_Activity                 0.027824
Gender_Male                      -0.002032
Peer_Influence_Neutral           -0.007795
School_Type_Public               -0.008844
Sleep_Hours                      -0.017022
Learning_Disabilities_Yes        -0.085066
Parental_Involvement                   NaN
Access_to_Resources                    NaN
Motivation_Level                       NaN
Family_Income                          NaN
Teacher_Quality                        NaN
Parental_Education_Level               NaN
Distance_from_Home                     NaN
Name: Exam_Score, dtype: float64


3.7 — Train/Test Split

In [26]:
X = df.drop('Exam_Score', axis=1)
y = df['Exam_Score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (5285, 20)
Test set: (1322, 20)


3.8 — Feature Scaling

In [27]:
scaler = StandardScaler()

# Fit ONLY on training data, then apply to both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for readability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

X_train_scaled.describe()

d:\student-exam-marks-prediction\venv\Lib\site-packages\sklearn\utils\extmath.py:1211: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
d:\student-exam-marks-prediction\venv\Lib\site-packages\sklearn\utils\extmath.py:1216: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
d:\student-exam-marks-prediction\venv\Lib\site-packages\sklearn\utils\extmath.py:1240: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Sleep_Hours,Previous_Scores,Motivation_Level,Tutoring_Sessions,Family_Income,Teacher_Quality,Physical_Activity,Parental_Education_Level,Distance_from_Home,Extracurricular_Activities_Yes,Internet_Access_Yes,School_Type_Public,Peer_Influence_Neutral,Peer_Influence_Positive,Learning_Disabilities_Yes,Gender_Male
count,5.285000e+03,5.285000e+03,0.0,0.0,5.285000e+03,5.285000e+03,0.0,5.285000e+03,0.0,0.0,5.285000e+03,0.0,0.0,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03
mean,1.962900e-16,1.687287e-16,NaN,NaN,-2.446902e-16,-2.732598e-16,NaN,9.814498e-17,NaN,NaN,-2.009955e-16,NaN,NaN,-5.512252e-17,-4.033355e-17,6.789481e-17,-5.108917e-17,-3.226684e-17,-1.445286e-17,8.066710e-17
std,1.000095e+00,1.000095e+00,NaN,NaN,1.000095e+00,1.000095e+00,NaN,1.000095e+00,NaN,NaN,1.000095e+00,NaN,NaN,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00
min,-2.679226e+00,-1.725428e+00,NaN,NaN,-2.074245e+00,-1.739730e+00,NaN,-1.314743e+00,NaN,NaN,-2.871410e+00,NaN,NaN,-1.220891e+00,-3.439138e+00,-1.516246e+00,-7.887717e-01,-8.287832e-01,-3.514426e-01,-1.169997e+00
25%,-6.653290e-01,-8.572273e-01,NaN,NaN,-7.025751e-01,-8.372767e-01,NaN,-4.025781e-01,NaN,NaN,-9.368824e-01,NaN,NaN,-1.220891e+00,2.907705e-01,-1.516246e+00,-7.887717e-01,-8.287832e-01,-3.514426e-01,-1.169997e+00
50%,5.969924e-03,1.097367e-02,NaN,NaN,-1.674034e-02,-4.242668e-03,NaN,-4.025781e-01,NaN,NaN,3.038142e-02,NaN,NaN,8.190739e-01,2.907705e-01,6.595237e-01,-7.887717e-01,-8.287832e-01,-3.514426e-01,8.547031e-01
75%,6.772688e-01,8.791746e-01,NaN,NaN,6.690944e-01,8.982109e-01,NaN,5.095871e-01,NaN,NaN,9.976452e-01,NaN,NaN,8.190739e-01,2.907705e-01,6.595237e-01,1.267794e+00,1.206588e+00,-3.514426e-01,8.547031e-01
max,2.691166e+00,1.747376e+00,NaN,NaN,2.040764e+00,1.731245e+00,NaN,1.877835e+00,NaN,NaN,2.932173e+00,NaN,NaN,8.190739e-01,2.907705e-01,6.595237e-01,1.267794e+00,1.206588e+00,2.845415e+00,8.547031e-01


3.9 — Save Processed Data

In [28]:
import os
import joblib
import json

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save scaling and mapping objects for the app
joblib.dump(scaler, '../models/scaler.pkl')
with open('../models/ordinal_mappings.json', 'w') as f:
    json.dump(ordinal_cols, f, indent=2)
training_columns = list(X_train.columns)
with open('../models/training_columns.json', 'w') as f:
    json.dump(training_columns, f, indent=2)

print("Processed data and preprocessing objects saved successfully.")


Processed data saved to data/processed/
